## Instal unsloth

In [1]:
!pip install --upgrade --force-reinstall --no-cache-dir torch==2.1.0 triton --index-url https://download.pytorch.org/whl/cu121
!pip install "unsloth[cu121-ampere] @ git+https://github.com/unslothai/unsloth.git"

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 GB 224.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.4/209.4 MB 244.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 159.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 236.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 213.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 172.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 227.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 260.7 MB/s eta 0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.9.0
    Uninstal

## Load the mistral 7b instruct model

In [2]:
from unsloth import FastLanguageModel

model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=5048,  # Maximum sequence length
    dtype=None,  # Use float16 for a balance between performance and precision
    load_in_4bit=True,   # Load in full precision for better accuracy
)

# Display some information about the model
print("\nModel Information:")
print(f"Model Name: {model_name}")
print(f"Vocabulary Size: {len(tokenizer)}")
print(f"Maximum Sequence Length: {model.config.max_position_embeddings}")

# Calculate and display the number of parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of Parameters: {num_params:,}")

/opt/conda/lib/python3.10/site-packages/unsloth/__init__.py:114: UserWarning: Unsloth: Running `ldconfig /usr/lib64-nvidia` to link CUDA.
  warnings.warn(


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2024-08-06 13:04:28.740410: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-06 13:04:28.740530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-06 13:04:28.842808: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


==((====))==  Unsloth 2024.8: Fast Mistral patching. Transformers = 4.43.4.
   \\   /|    GPU: Tesla P100-PCIE-16GB. Max memory: 15.888 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.1.0+cu121. CUDA = 6.0. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.22.post7. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Unsloth: Will load unsloth/mistral-7b-instruct-v0.3-bnb-4bit as a legacy tokenizer.



Model Information:
Model Name: unsloth/mistral-7b-instruct-v0.3-bnb-4bit
Vocabulary Size: 32768
Maximum Sequence Length: 32768
Number of Parameters: 3,758,362,624


## Make a first test without fine-tuning

In [3]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [4]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is (are) Glaucoma ?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1000)

<s>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is (are) Glaucoma ?

### Input:


### Response:
Glaucoma is a group of eye conditions that damage the optic nerve, which transmits visual information from the eye to the brain. It is often associated with an increase in intraocular pressure, although this is not always the case. There are several types of glaucoma, including open-angle glaucoma, angle-closure glaucoma, congenital glaucoma, and secondary glaucoma. Symptoms can include loss of peripheral vision, tunnel vision, halos around lights, and eye pain or redness. If left untreated, glaucoma can lead to permanent vision loss.</s>


## Load the dataset

In [5]:
import pandas as pd
from datasets import Dataset

# Load the CSV file
data = pd.read_csv('/kaggle/input/layoutlm/medquad.csv')

# Keep only the first 100 questions/answers for time constraints
data = data.iloc[:100, :2]
data.columns = ['question', 'answer']

# Convert the DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(data)

# Preprocessing function to format data according to the alpaca_prompt format
def preprocess_function(examples):
    alpaca_prompts = [f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{question}\n\n### Input:\n{input_text}\n\n### Response:\n" for question, input_text in zip(examples['question'], examples['answer'])]
    examples['text'] = [prompt + response for prompt, response in zip(alpaca_prompts, examples['answer'])]
    return examples

# Apply the preprocessing function
dataset = dataset.map(preprocess_function, batched=True, remove_columns=['question', 'answer'])

# Split the dataset into training and validation sets (80% train, 20% validation)
train_test_split = dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']
print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(eval_dataset)}")

# Check the first few lines of the preprocessed dataset
print("\nExample from the training set:")
print(train_dataset[0])
print("\nExample from the validation set:")
print(eval_dataset[0])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Training set size: 80
Validation set size: 20

Example from the training set:
{'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat are the symptoms of Problems with Taste ?\n\n### Input:\nSymptoms Vary With Disorders There are several types of taste disorders depending on how the sense of taste is affected. People who have taste disorders usually lose their ability to taste or can no longer perceive taste in the same way. True taste disorders are rare. Most changes in the perception of food flavor result from the loss of smell.  Phantom Taste Perception. The most common taste complaint is "phantom taste perception" -- tasting something when nothing is in the mouth.  Hypogeusia. Some people have hypogeusia, or the reduced ability to taste sweet, sour, bitter, salty, and savory, or umami. This disorder is usually temporary. Dysgeusia. Dysgeusia is a

In [6]:
from transformers import TrainingArguments
import torch

# Configurer les arguments d'entraînement
training_args = TrainingArguments(
   per_device_train_batch_size = 4,
   gradient_accumulation_steps = 4,
   warmup_steps = 5,
   num_train_epochs=3,
   max_steps=-1,
   learning_rate = 2e-4,
   fp16 = not torch.cuda.is_bf16_supported(),
   bf16 = torch.cuda.is_bf16_supported(),
   logging_steps = 1,
   optim = "adamw_8bit",
   weight_decay = 0.01,
   lr_scheduler_type = "linear",
   seed = 3407,
   output_dir = "outputs",
   report_to = "none",
)

#Do model patching and add fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = training_args,
)

Unsloth 2024.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
#@title Show current memory stats
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla P100-PCIE-16GB. Max memory = 15.888 GB.
4.924 GB of memory reserved.


## Train the model

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 100 | Num Epochs = 3
O^O/ \_/ \    Batch size per device = 4 | Gradient Accumulation steps = 4
\        /    Total batch size = 16 | Total steps = 18
 "-____-"     Number of trainable parameters = 20,971,520


Step,Training Loss
1,0.867000
2,1.106300
3,0.643500
4,0.727200
5,0.675300
6,0.675800
7,0.683800
8,0.528600
9,0.601900
10,0.538100


The model seems to learn, but of course we need more epochs and, thus, more time !

**END**